# Chapter 9 Lab — Momentum and Oscillators

**Purpose:** replicate and understand the mechanics of the main Chapter 9 ideas.

This notebook is deliberately **not** an optimisation exercise and **not** a production backtest.
We use deterministic synthetic price data so the learning objective stays focused on indicator behaviour.


## Learning objectives

By the end of this notebook you should be able to:
1. calculate n-period momentum;
2. calculate RSI(14) and RSI(2);
3. calculate Stochastic %K and %D;
4. calculate MACD;
5. explain why faster signals are noisier;
6. build one simple **trend + oscillator** entry condition;
7. describe when each idea can fail.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

np.random.seed(7)

n1, n2, n3 = 120, 90, 110
up = 100 + np.cumsum(0.22 + np.random.normal(0, 0.75, n1))
side = up[-1] + np.cumsum(np.random.normal(0, 0.90, n2))
down = side[-1] + np.cumsum(-0.20 + np.random.normal(0, 0.80, n3))

close = np.concatenate([up, side, down])
idx = pd.date_range('2025-01-01', periods=len(close), freq='B')

df = pd.DataFrame({'Close': close}, index=idx)
spread = np.abs(np.random.normal(1.2, 0.35, len(df)))
df['High'] = df['Close'] + spread
df['Low'] = df['Close'] - spread
df.head()


In [ ]:
fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(df.index, df['Close'], label='Synthetic close')
ax.set_title('Synthetic price series with changing regimes')
ax.set_ylabel('Price')
ax.legend()
plt.show()


## 1. n-period momentum

`M_t(n) = P_t - P_{t-n}`

Momentum is positive when price is above its value n periods ago and negative when it is below.
A key point: **price can still be rising while momentum is already falling**.


In [ ]:
def momentum(series, n=20):
    return series - series.shift(n)

df['Momentum20'] = momentum(df['Close'], 20)

fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(df.index, df['Momentum20'], label='20-period momentum')
ax.axhline(0, linewidth=1)
ax.set_title('20-period momentum')
ax.legend()
plt.show()


## 2. RSI

`RSI = 100 - 100 / (1 + RS)`

Compare RSI(14) with RSI(2): the 2-period version should reach extremes much more frequently.


In [ ]:
def rsi_wilder(series, n=14):
    delta = series.diff()
    gain = delta.clip(lower=0)
    loss = -delta.clip(upper=0)
    avg_gain = gain.ewm(alpha=1/n, adjust=False, min_periods=n).mean()
    avg_loss = loss.ewm(alpha=1/n, adjust=False, min_periods=n).mean()
    rs = avg_gain / avg_loss.replace(0, np.nan)
    rsi = 100 - (100 / (1 + rs))
    return rsi.fillna(100).where(avg_gain.notna())

df['RSI14'] = rsi_wilder(df['Close'], 14)
df['RSI2'] = rsi_wilder(df['Close'], 2)

fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(df.index, df['RSI14'], label='RSI(14)')
ax.plot(df.index, df['RSI2'], label='RSI(2)', alpha=0.75)
ax.axhline(70, linewidth=1)
ax.axhline(30, linewidth=1)
ax.set_ylim(0, 100)
ax.set_title('RSI(14) versus RSI(2)')
ax.legend()
plt.show()


## 3. Stochastic oscillator

`%K = 100 * (C - L_n) / (H_n - L_n)`

Momentum asks **how far price moved**. Stochastic asks **where the close sits inside the recent range**.


In [ ]:
def stochastic_k(high, low, close, n=14):
    lowest = low.rolling(n).min()
    highest = high.rolling(n).max()
    denom = (highest - lowest).replace(0, np.nan)
    return 100 * (close - lowest) / denom

df['StochK14'] = stochastic_k(df['High'], df['Low'], df['Close'], 14)
df['StochD14'] = df['StochK14'].rolling(3).mean()

fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(df.index, df['StochK14'], label='%K')
ax.plot(df.index, df['StochD14'], label='%D')
ax.axhline(80, linewidth=1)
ax.axhline(20, linewidth=1)
ax.set_ylim(0, 100)
ax.set_title('14-period Stochastic')
ax.legend()
plt.show()


## 4. MACD

`MACD = EMA_fast - EMA_slow`

`Signal = EMA(MACD)`

`Histogram = MACD - Signal`


In [ ]:
def macd(series, fast=12, slow=26, signal=9):
    ema_fast = series.ewm(span=fast, adjust=False).mean()
    ema_slow = series.ewm(span=slow, adjust=False).mean()
    line = ema_fast - ema_slow
    signal_line = line.ewm(span=signal, adjust=False).mean()
    hist = line - signal_line
    return line, signal_line, hist

df['MACD'], df['MACDSignal'], df['MACDHist'] = macd(df['Close'])

fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(df.index, df['MACD'], label='MACD')
ax.plot(df.index, df['MACDSignal'], label='Signal')
ax.axhline(0, linewidth=1)
ax.set_title('MACD')
ax.legend()
plt.show()


## 5. Trend + oscillator architecture

Illustrative rule only:
- trend filter: `Close > MA(40)`;
- short-term pullback: `Stochastic(5) < 20`.

This demonstrates **long-term direction + short-term timing**. It is not a complete strategy.


In [ ]:
df['MA40'] = df['Close'].rolling(40).mean()
df['StochK5'] = stochastic_k(df['High'], df['Low'], df['Close'], 5)
df['LongEntryDemo'] = (df['Close'] > df['MA40']) & (df['StochK5'] < 20)

signals = df.index[df['LongEntryDemo']]

fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(df.index, df['Close'], label='Close')
ax.plot(df.index, df['MA40'], label='MA(40)')
ax.scatter(signals, df.loc[signals, 'Close'], marker='^', label='Illustrative long entry')
ax.set_title('Long-term trend + short-term oscillator timing')
ax.legend()
plt.show()

print('Illustrative entry signals:', int(df['LongEntryDemo'].sum()))


## 6. What is missing before this becomes a strategy?

- exact execution timing;
- exit rule;
- stop / invalidation rule;
- position sizing;
- transaction costs and slippage;
- market universe;
- out-of-sample validation;
- robustness testing.

That is why this notebook stops here.


In [ ]:
assert df['RSI14'].dropna().between(0, 100).all()
assert df['RSI2'].dropna().between(0, 100).all()
assert df['StochK14'].dropna().between(0, 100).all()
assert df['StochK5'].dropna().between(0, 100).all()
print('Basic indicator sanity checks passed.')


## Reflection — answer before Chapter 10

1. Why can momentum weaken while price is still rising?
2. What information does RSI use that simple n-period momentum does not?
3. Why is Stochastic not the same as momentum?
4. What exactly is MACD measuring?
5. Why can RSI/Stochastic extremes fail in persistent trends?
6. Why is `trend filter + oscillator trigger` more coherent than blindly stacking indicators?
7. Which of RSI, Stochastic and MACD are likely to contain overlapping information?
8. What additional components are required before the final example becomes a tradable system?

**Do not optimise any parameters yet.**
